# API Lluvias

Importé las librerías necesarias para trabajar con tablas, hacer solicitudes a páginas web y APIs.

In [1]:
import pandas as pd
import requests
import io
import time

Descargué la página de Wikipedia usando requests y un User-Agent para que la solicitud sea identificada como si viniera de un navegador. Luego revisé el código de respuesta para comprobar que la página se cargó correctamente.

In [2]:
url = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"

headers = {
    "User-Agent": "Mozilla/5.0"
}

respuesta = requests.get(url, headers=headers)

print(respuesta.status_code)

200


Utilicé pd.read_html para extraer las tablas disponibles en la página. Luego conté cuántas tablas encontró Pandas para identificar cuál contiene los departamentos y sus capitales.

In [3]:
tablas = pd.read_html(
    io.StringIO(respuesta.text)
)

len(tablas)

6

Revisé las primeras filas de cada tabla encontrada para identificar cuál contiene la información de departamentos y capitales.

In [4]:
for i, tabla in enumerate(tablas):
    print("Tabla", i)
    print(tabla.head())
    print()

Tabla 0
  Departamentos del Perú (con las provincias de régimen especial)
0  Madre de Dios Cusco San Martín Tumbes Lambayeq...             

Tabla 1
   Bandera  Escudo Departamento                          Nombre oficial  \
0      NaN     NaN     Amazonas                Departamento de Amazonas   
1      NaN     NaN       Áncash                  Departamento de Áncash   
2      NaN     NaN     Apurímac  Departamento de Apurímac Apurimaq suyu   
3      NaN     NaN     Arequipa   Departamento de Arequipa Ariqipa suyu   
4      NaN     NaN     Ayacucho                Departamento de Ayacucho   

  Abreviatura Nacional      Capital Ciudad más poblada Superficie (km²)  \
0                  AMA  Chachapoyas       Bagua Grande           39 200   
1                  ANC       Huaraz           Chimbote           35 900   
2                  APU      Abancay        Andahuaylas           20 900   
3                  ARE     Arequipa           Arequipa           63 300   
4                  AYA   

Seleccioné la tabla que contiene la información de los departamentos del Perú y sus capitales, y conservé únicamente esas dos columnas porque son las necesarias para la siguiente parte del análisis.

In [5]:
departamentos = tablas[1][["Departamento", "Capital"]].copy()

departamentos.head()

,Departamento,Capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho


Como la consigna dice que que al final deben quedar 25 filas: 24 departamentos y el Callao, verificamos ello:

In [6]:
departamentos.shape

(24, 2)

In [7]:
departamentos["Departamento"].tolist()

['Amazonas',
 'Áncash',
 'Apurímac',
 'Arequipa',
 'Ayacucho',
 'Cajamarca',
 'Cusco',
 'Huancavelica',
 'Huánuco',
 'Ica',
 'Junín',
 'La Libertad',
 'Lambayeque',
 'Lima',
 'Loreto',
 'Madre de Dios',
 'Moquegua',
 'Pasco',
 'Piura',
 'Puno',
 'San Martín',
 'Tacna',
 'Tumbes',
 'Ucayali']

Como la tabla extraída inicialmente contenía 24 filas y no incluía al Callao. Como la consigna exige trabajar con los 24 departamentos y la Provincia Constitucional del Callao, agregué manualmente una fila con Callao como departamento y capital.

In [8]:
callao = pd.DataFrame({
    "Departamento": ["Callao"],
    "Capital": ["Callao"]
})

departamentos = pd.concat(
    [departamentos, callao],
    ignore_index=True
)

departamentos.shape

(25, 2)

Ahora que ya tenemos las 25 filas, toca revisar la columna Capital, porque la consigna avisa que algunas celdas pueden traer texto extra y eso puede hacer que Open-Meteo no encuentre bien la ciudad.Entonces:

In [9]:
departamentos[["Departamento", "Capital"]]

,Departamento,Capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


In [10]:
departamentos[
    departamentos["Capital"].astype(str).str.contains(r"\(|\)|de facto|sede", case=False, regex=True, na=False)
]

,Departamento,Capital
13,Lima,Huacho (de facto)


Al revisar las capitales encontré que Lima aparecía como Huacho (de facto). Para realizar correctamente la búsqueda en la API eliminé el texto adicional y mantuve Huacho como nombre de la ciudad.

In [11]:
departamentos.loc[
    departamentos["Departamento"] == "Lima",
    "Capital"
] = "Huacho"

Verificamos:

In [12]:
departamentos[
    departamentos["Departamento"] == "Lima"
]

,Departamento,Capital
13,Lima,Huacho


También conviene comprobar nuevamente toda la tabla:

In [13]:
departamentos

,Departamento,Capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


Ahora viene la geocodificación con Open-Meteo.Pero, primero probamos con una sola ciudad, por ejemplo Chachapoyas.Además, pedí hasta 10 resultados para poder revisar si aparecen varias ciudades con el mismo nombre.

In [14]:
capital = "Chachapoyas"

url_geo = "https://geocoding-api.open-meteo.com/v1/search"

params = {
    "name": capital,
    "count": 10,
    "language": "es"
}

respuesta_geo = requests.get(url_geo, params=params)
datos_geo = respuesta_geo.json()

datos_geo

{'results': [{'id': 3698540,
   'name': 'Chachapoyas',
   'latitude': -6.23169,
   'longitude': -77.86903,
   'elevation': 2360.0,
   'feature_code': 'PPLA',
   'country_code': 'PE',
   'admin1_id': 3699699,
   'admin2_id': 3698538,
   'admin3_id': 8350561,
   'timezone': 'America/Lima',
   'population': 32026,
   'country_id': 3932488,
   'country': 'Perú',
   'admin1': 'Departamento de Amazonas',
   'admin2': 'Provincia de Chachapoyas',
   'admin3': 'Chachapoyas'},
  {'id': 3859420,
   'name': 'Destilería Chachapoyas',
   'latitude': -24.74735,
   'longitude': -65.38527,
   'elevation': 1252.0,
   'feature_code': 'PPL',
   'country_code': 'AR',
   'admin1_id': 3838231,
   'admin2_id': 3862749,
   'timezone': 'America/Argentina/Salta',
   'country_id': 3865483,
   'country': 'Argentina',
   'admin1': 'Provincia de Salta',
   'admin2': 'Departamento Capital'},
  {'id': 13255311,
   'name': 'Chachapoyas',
   'latitude': -8.97357,
   'longitude': -78.49754,
   'elevation': 156.0,
   'fea

Al parecer la prueba salió bien: para Chachapoyas, Open-Meteo devolvió como primer resultado una ciudad de Perú y admin1 = "Departamento de Amazonas", que corresponde correctamente al departamento Amazonas.

Ahora vamos a automatizar las 25 capitales, pero comparando bien los nombres porque la API puede devolver cosas como Departamento de Amazonas, Ancash sin tilde o Provincia Constitucional del Callao.

In [15]:
import unicodedata

def normalizar(texto):
    texto = str(texto).lower().strip()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )
    
    texto = texto.replace("departamento de ", "")
    texto = texto.replace("provincia constitucional del ", "")
    
    return texto

Creé una función para normalizar los nombres de los departamentos. La función convierte el texto a minúsculas, elimina tildes y quita expresiones como Departamento de o Provincia Constitucional del, para poder comparar correctamente los nombres que devuelve la API.

Ahora crearemos una función que consulta la API de Open-Meteo y revisa hasta 10 resultados por capital. Solo selecciona un resultado si pertenece a Perú y si el departamento devuelto por la API coincide con el departamento de nuestra tabla.

In [16]:
def buscar_capital(capital, departamento):
    url_geo = "https://geocoding-api.open-meteo.com/v1/search"
    
    params = {
        "name": capital,
        "count": 10,
        "language": "es"
    }
    
    respuesta = requests.get(url_geo, params=params)
    datos = respuesta.json()
    
    resultados = datos.get("results", [])
    
    for resultado in resultados:
        pais = resultado.get("country_code")
        admin1 = resultado.get("admin1", "")
        
        if (
            pais == "PE"
            and normalizar(admin1) == normalizar(departamento)
        ):
            return {
                "admin1_api": admin1,
                "latitud": resultado.get("latitude"),
                "longitud": resultado.get("longitude")
            }
    
    return {
        "admin1_api": None,
        "latitud": None,
        "longitud": None
    }

Ahora recorremos las 25 capitales:

In [17]:
resultados_geo = []

for _, fila in departamentos.iterrows():
    
    resultado = buscar_capital(
        fila["Capital"],
        fila["Departamento"]
    )
    
    resultados_geo.append(resultado)
    
    time.sleep(1)

Consulté la API para cada una de las 25 capitales y guardé el departamento devuelto por Open-Meteo, la latitud y la longitud. Agregué una pausa de un segundo entre consultas para no enviar demasiadas solicitudes seguidas.

In [18]:
geo_df = pd.DataFrame(resultados_geo)

departamentos_geo = pd.concat(
    [
        departamentos.reset_index(drop=True),
        geo_df
    ],
    axis=1
)

departamentos_geo

,Departamento,Capital,admin1_api,latitud,longitud
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422


Uní los resultados de la geocodificación con la tabla original para poder revisar, fila por fila, si cada capital fue ubicada correctamente.

Finalmente, verificamos si alguna capital quedó sin coordenadas:

In [19]:
departamentos_geo[
    departamentos_geo["latitud"].isna()
]

,Departamento,Capital,admin1_api,latitud,longitud


Revisamos:

In [20]:
departamentos_geo[
    [
        "Departamento",
        "Capital",
        "admin1_api",
        "latitud",
        "longitud"
    ]
]

,Departamento,Capital,admin1_api,latitud,longitud
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422


Ahora toca la siguiente parte: descargar la lluvia diaria para cada capital entre el 1 de enero y el 31 de mayo de 2024 usando la API histórica de Open-Meteo.

Primero prueba con una sola ciudad, por ejemplo Chachapoyas:

In [21]:
lat = departamentos_geo.loc[0, "latitud"]
lon = departamentos_geo.loc[0, "longitud"]

url_lluvia = "https://archive-api.open-meteo.com/v1/archive"

params_lluvia = {
    "latitude": lat,
    "longitude": lon,
    "start_date": "2024-01-01",
    "end_date": "2024-05-31",
    "daily": "precipitation_sum",
    "timezone": "America/Lima"
}

respuesta_lluvia = requests.get(url_lluvia, params=params_lluvia)
datos_lluvia = respuesta_lluvia.json()

datos_lluvia

{'latitude': -6.221441,
 'longitude': -77.88461,
 'generationtime_ms': 1.710057258605957,
 'utc_offset_seconds': -18000,
 'timezone': 'America/Lima',
 'timezone_abbreviation': 'GMT-5',
 'elevation': 2361.0,
 'daily_units': {'time': 'iso8601', 'precipitation_sum': 'mm'},
 'daily': {'time': ['2024-01-01',
   '2024-01-02',
   '2024-01-03',
   '2024-01-04',
   '2024-01-05',
   '2024-01-06',
   '2024-01-07',
   '2024-01-08',
   '2024-01-09',
   '2024-01-10',
   '2024-01-11',
   '2024-01-12',
   '2024-01-13',
   '2024-01-14',
   '2024-01-15',
   '2024-01-16',
   '2024-01-17',
   '2024-01-18',
   '2024-01-19',
   '2024-01-20',
   '2024-01-21',
   '2024-01-22',
   '2024-01-23',
   '2024-01-24',
   '2024-01-25',
   '2024-01-26',
   '2024-01-27',
   '2024-01-28',
   '2024-01-29',
   '2024-01-30',
   '2024-01-31',
   '2024-02-01',
   '2024-02-02',
   '2024-02-03',
   '2024-02-04',
   '2024-02-05',
   '2024-02-06',
   '2024-02-07',
   '2024-02-08',
   '2024-02-09',
   '2024-02-10',
   '2024-02-11'

Hice una consulta de prueba a la API histórica de Open-Meteo usando las coordenadas de una capital. Solicité la precipitación diaria entre el 1 de enero y el 31 de mayo de 2024.

Debería darnos alrededor de 152 días, porque ese periodo va del 1 de enero al 31 de mayo de 2024.

In [22]:
lluvias = datos_lluvia["daily"]["precipitation_sum"]

len(lluvias)

152

Ahora revisamos cuántos valores faltantes hay:

In [23]:
sum(valor is None for valor in lluvias)

0

Y calculamos:

In [24]:
lluvias_validas = [
    valor for valor in lluvias
    if valor is not None
]

lluvia_total = sum(lluvias_validas)

dias_lluvia_fuerte = sum(
    valor >= 20
    for valor in lluvias_validas
)

print("Lluvia total:", lluvia_total)
print("Días con lluvia fuerte:", dias_lluvia_fuerte)

Lluvia total: 455.0
Días con lluvia fuerte: 0


Revisé los datos diarios de precipitación y conté los valores faltantes. Luego calculé la lluvia total acumulada y el número de días con precipitación igual o mayor a 20 mm.

Ahora crearé una función que consulta la API histórica de Open-Meteo para cada capital entre enero y mayo de 2024. La función calcula la precipitación total, cuenta los días con 20 mm o más de lluvia y registra cuántos valores faltantes recibió.

In [25]:
def obtener_lluvia(latitud, longitud):
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitud,
        "longitude": longitud,
        "start_date": "2024-01-01",
        "end_date": "2024-05-31",
        "daily": "precipitation_sum",
        "timezone": "America/Lima"
    }

    respuesta = requests.get(url, params=params)
    datos = respuesta.json()

    lluvias = datos["daily"]["precipitation_sum"]

    cantidad_none = sum(
        valor is None
        for valor in lluvias
    )

    lluvias_validas = [
        valor for valor in lluvias
        if valor is not None
    ]

    lluvia_total = sum(lluvias_validas)

    dias_fuertes = sum(
        valor >= 20
        for valor in lluvias_validas
    )

    return lluvia_total, dias_fuertes, cantidad_none

Ahora recorrer las 25 capitales. Incluí una pausa de un segundo entre solicitudes para evitar realizar demasiadas consultas seguidas a la API.

In [26]:
resultados_lluvia = []

for _, fila in departamentos_geo.iterrows():

    lluvia_total, dias_fuertes, cantidad_none = obtener_lluvia(
        fila["latitud"],
        fila["longitud"]
    )

    resultados_lluvia.append({
        "lluvia_total_mm": lluvia_total,
        "dias_lluvia_fuerte": dias_fuertes,
        "valores_none": cantidad_none
    })

    time.sleep(1)

Unimos Uní los resultados de precipitación con la tabla de departamentos y coordenadas para tener toda la información en una sola base.

In [27]:
lluvias_df = pd.DataFrame(resultados_lluvia)

resultado_final = pd.concat(
    [
        departamentos_geo.reset_index(drop=True),
        lluvias_df
    ],
    axis=1
)

resultado_final.head()

,Departamento,Capital,admin1_api,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,valores_none
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903,455.0,0,0
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869,1058.3,3,0
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380,670.6,2,0
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747,235.3,1,0
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345,467.5,1,0


VVerificamos los None

In [28]:
resultado_final[
    [
        "Departamento",
        "Capital",
        "valores_none"
    ]
]

,Departamento,Capital,valores_none
0,Amazonas,Chachapoyas,0
1,Áncash,Huaraz,0
2,Apurímac,Abancay,0
3,Arequipa,Arequipa,0
4,Ayacucho,Ayacucho,0
5,Cajamarca,Cajamarca,0
6,Cusco,Cusco,0
7,Huancavelica,Huancavelica,0
8,Huánuco,Huánuco,0
9,Ica,Ica,0


Y para saber el total:

In [29]:
resultado_final["valores_none"].sum()

np.int64(0)

Luego revisemos que la tabla final tenga las columnas correctas:

In [30]:
resultado_final[
    [
        "Departamento",
        "Capital",
        "latitud",
        "longitud",
        "lluvia_total_mm",
        "dias_lluvia_fuerte"
    ]
].head()

,Departamento,Capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,Amazonas,Chachapoyas,-6.23169,-77.86903,455.0,0
1,Áncash,Huaraz,-9.52614,-77.52869,1058.3,3
2,Apurímac,Abancay,-13.63426,-72.88380,670.6,2
3,Arequipa,Arequipa,-16.39899,-71.53747,235.3,1
4,Ayacucho,Ayacucho,-13.16380,-74.22345,467.5,1


In [31]:
resultado_final.shape

(25, 8)

Para el archivo final vamos a quedarnos solo con las 6 columnas que pide la consigna:

In [32]:
lluvias_final = resultado_final[
    [
        "Departamento",
        "Capital",
        "latitud",
        "longitud",
        "lluvia_total_mm",
        "dias_lluvia_fuerte"
    ]
].copy()

In [33]:
lluvias_final.columns = [
    "departamento",
    "capital",
    "latitud",
    "longitud",
    "lluvia_total_mm",
    "dias_lluvia_fuerte"
]

Verificamos:

In [34]:
lluvias_final.shape

(25, 6)

Listo, terminamos. Ahora, guardamos la tabla final en datos/lluvias_por_departamento.csv, con las seis columnas solicitadas y codificación UTF-8 para conservar correctamente las tildes.

In [35]:
import os

os.makedirs("datos", exist_ok=True)

lluvias_final.to_csv(
    "datos/lluvias_por_departamento.csv",
    index=False,
    encoding="utf-8-sig"
)

Comprobamos

In [36]:
os.path.exists("datos/lluvias_por_departamento.csv")

True